##### 3.3 Labelling Strategy
Market stress is defined using a rule-based approach following Holló et al. (2012) and Kritzman & Li (2010).

Let $r_t = \log(C_t) - \log(C_{t-1})$ denote the daily log return and $\sigma_t^{14} = \text{std}(r_{t-13}, \dots, r_t)$ the 14-day rolling volatility.

Using percentiles computed on the training set only to avoid look-ahead bias, we define:

- $q_{1\%}^r = P_1(r)$ (1st percentile of returns)
- $q_{90\%}^{\sigma} = P_{90}(\sigma^{14})$ (90th percentile of volatility)

A day is classified as stress if either:

$$
\text{stress}_t = \begin{cases} 
1 & \text{if } r_t \le q_{1\%}^r \text{ OR } \sigma_t^{14} \ge q_{90\%}^{\sigma} \\
0 & \text{otherwise}
\end{cases}
$$

This captures both acute price shocks (extreme negative returns) and sustained uncertainty regimes (elevated volatility).

In [ ]:
import pandas as pd
import numpy as np
from typing import Literal

# =============================================================================
# CONFIG
# =============================================================================

INPUT_PATH  = "../data/processed/ES_D.csv"
OUTPUT_PATH = "../data/processed/ES_labeled_debug.csv"

RET_Q = 0.01      # 1st percentile
VOL_Q = 0.90      # 90th percentile
VOL_WINDOW = 14

# =============================================================================
# LOAD ES DATA
# =============================================================================

df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

# =============================================================================
# BASE SERIES (LEAKAGE-SAFE)
# =============================================================================

df["ret"] = np.log(df["close"]).diff()
df["vol_14d"] = df["ret"].rolling(VOL_WINDOW).std()

# =============================================================================
# CORE LABEL GENERATOR (USE INSIDE EACH WALK-FORWARD FOLD)
# =============================================================================

def _horizon_any_future(stress: pd.Series, horizon: int) -> pd.Series:
    """
    For each t, return max(stress[t+1], ..., stress[t+horizon]).
    No leakage: does not include stress[t] or any past terms.
    """
    if horizon < 1:
        raise ValueError("horizon must be >= 1")

    shifted = pd.concat([stress.shift(-i) for i in range(1, horizon + 1)], axis=1)
    return shifted.max(axis=1)


def generate_stress_targets_for_fold(
    es_df: pd.DataFrame,
    train_end: pd.Timestamp,
    horizon: Literal[1, 3, 5],
    ret_q: float = RET_Q,
    vol_q: float = VOL_Q,
) -> pd.Series:
    """
    Compute fold-specific thresholds from TRAIN ONLY, then produce horizon target:
      stress_h(t) = max(stress(t+1..t+horizon))

    Parameters
    ----------
    es_df : DataFrame containing columns: date, ret, vol_14d
    train_end : last date included in the training window for this fold
    horizon : forecast horizon (1/3/5)
    ret_q : return percentile threshold (e.g. 0.01)
    vol_q : vol percentile threshold (e.g. 0.90)

    Returns
    -------
    pd.Series (0/1) aligned with es_df rows (NaN near end due to shifting).
    """

    # ---- training slice for threshold estimation only ----
    train_mask = es_df["date"] <= train_end
    train = es_df.loc[train_mask].dropna(subset=["ret", "vol_14d"])

    if train.empty:
        raise ValueError("Training window has no valid ret/vol_14d rows (check dates/window).")

    q_ret = train["ret"].quantile(ret_q)
    q_vol = train["vol_14d"].quantile(vol_q)

    # ---- base stress_t (defined for all dates, using fold thresholds) ----
    stress_t = ((es_df["ret"] <= q_ret) | (es_df["vol_14d"] >= q_vol)).astype(int)

    # ---- horizon target: any stress in next H days ----
    stress_h = _horizon_any_future(stress_t, horizon=horizon)

    return stress_h


# =============================================================================
# DEBUG OUTPUT (NOT FOR MODEL TRAINING)
# =============================================================================
# This block is ONLY to sanity-check that the label mechanics work end-to-end.
# In actual CV training, call generate_stress_targets_for_fold() inside each fold.

if __name__ == "__main__":
    # Example: treat all data as "train" just to generate an inspection file
    train_end = df["date"].max()

    df["stress_1d"] = generate_stress_targets_for_fold(df, train_end=train_end, horizon=1)
    df["stress_3d"] = generate_stress_targets_for_fold(df, train_end=train_end, horizon=3)
    df["stress_5d"] = generate_stress_targets_for_fold(df, train_end=train_end, horizon=5)

    out = df[["date", "ret", "vol_14d", "stress_1d", "stress_3d", "stress_5d"]]
    out.to_csv(OUTPUT_PATH, index=False)

    print("Saved debug labels →", OUTPUT_PATH)
    print(out.head(5))

In [ ]:
df.loc[
    (df["date"] >= "2025-01-01") & (df["date"] <= "2025-06-30"),
    ["date", "ret", "vol_14d", "stress_1d", "stress_3d", "stress_5d"]
]

In [ ]:
counts = {
    "stress_1d": df["stress_1d"].sum(),
    "stress_3d": df["stress_3d"].sum(),
    "stress_5d": df["stress_5d"].sum(),
}

ratios = {
    "stress_1d": df["stress_1d"].mean(),
    "stress_3d": df["stress_3d"].mean(),
    "stress_5d": df["stress_5d"].mean(),
}

print("Counts:")
for k, v in counts.items():
    print(f"{k}: {int(v)}")

print("\nRatios:")
for k, v in ratios.items():
    print(f"{k}: {v:.3f}")


In [ ]:
import pandas as pd
import os

# =============================================================================
# CONFIG
# =============================================================================

PATHS = {
    "ES":        "../data/processed/ES_D.csv",
    "DXY":       "../data/processed/DXY_D.csv",
    "VIX":       "../data/processed/VIX_D.csv",
    "MACRO":     "../data/processed/macro_D.csv",
    "SENTIMENT": "../data/processed/sentiment_analysis/daily_sentiment_features.csv",
}

OUTPUT_PATH = "../data/processed/dataset_merged.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# =============================================================================
# LOAD DATA
# =============================================================================

es        = pd.read_csv(PATHS["ES"], parse_dates=["date"])
dxy       = pd.read_csv(PATHS["DXY"], parse_dates=["date"])
vix       = pd.read_csv(PATHS["VIX"], parse_dates=["date"])
macro     = pd.read_csv(PATHS["MACRO"], parse_dates=["date"])
sentiment = pd.read_csv(PATHS["SENTIMENT"], parse_dates=["date"])

# Ensure sorted dates
for df in [es, dxy, vix, macro, sentiment]:
    df.sort_values("date", inplace=True)

# =============================================================================
# MERGE (ES AS MASTER CALENDAR)
# =============================================================================

df = es.copy()

df = df.merge(
    dxy,
    on="date",
    how="left",
    suffixes=("", "_DXY"),
)

df = df.merge(
    vix,
    on="date",
    how="left",
    suffixes=("", "_VIX"),
)

df = df.merge(
    macro,
    on="date",
    how="left",
)

df = df.merge(
    sentiment,
    on="date",
    how="left",
)

# =============================================================================
# FINAL CHECKS + SAVE
# =============================================================================

df = df.sort_values("date").reset_index(drop=True)

df.to_csv(OUTPUT_PATH, index=False)

print(f"Merged dataset saved → {OUTPUT_PATH}")
print(f"Shape: {df.shape}")
print("Missing values (top 10):")
print(df.isna().mean().sort_values(ascending=False).head(10))

In [ ]:
import pandas as pd
import numpy as np
import os

# =============================================================================
# CONFIG
# =============================================================================

INPUT_PATH  = "../data/processed/dataset_merged.csv"
OUTPUT_PATH = "../data/processed/dataset_features_base.csv"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# =============================================================================
# LOAD
# =============================================================================

df = pd.read_csv(INPUT_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)

# =============================================================================
# ES FEATURES (LEVEL / SAME-DAY ONLY)
# =============================================================================

df["es_close"] = df["close"]
df["es_volume"] = df["volume"]

# Intraday range (allowed: uses shift, not diff/rolling)
df["es_range"] = (df["high"] - df["low"]) / df["close"].shift(1)

# =============================================================================
# DXY FEATURES (LEVEL ONLY)
# =============================================================================

df["dxy_close"] = df["close_DXY"]

# =============================================================================
# VIX FEATURES (LEVEL ONLY)
# =============================================================================

df["vix_close"] = df["close_VIX"]

# =============================================================================
# MACRO FEATURES (LEVEL ONLY)
# =============================================================================

macro_cols = [
    "FED_RATE_USD",
    "US10Y",
    "US2Y",
    "US3M",
    "YIELD_CURVE_SLOPE",
    "TED_SPREAD",
    "BBB_SPREAD",
    "T10Y_IE",
]

for col in macro_cols:
    df[col] = df[col]

# =============================================================================
# SENTIMENT FEATURES (LEVEL ONLY)
# =============================================================================

sentiment_cols = [
    "sentiment_mean",
    "sentiment_median",
    "sentiment_std",
    "sentiment_min",
    "pct_negative",
    "article_count",
    "stress_article_count",
    "calm_article_count",
    "stress_ratio",
]

for col in sentiment_cols:
    df[col] = df[col]

# =============================================================================
# SELECT FINAL FEATURE SET
# =============================================================================

feature_cols = (
    ["date"] +
    ["es_close", "es_volume", "es_range"] +
    ["dxy_close"] +
    ["vix_close"] +
    macro_cols +
    sentiment_cols
)

df_out = df[feature_cols]

# =============================================================================
# SAVE
# =============================================================================

df_out.to_csv(OUTPUT_PATH, index=False)

print(f"Base feature dataset saved → {OUTPUT_PATH}")
print(f"Shape: {df_out.shape}")
print("Missing values (top 10):")
print(df_out.isna().mean().sort_values(ascending=False).head(10))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# ─────────────────────────────────────────────
#  GLOBAL STYLE  —  light academic palette
# ─────────────────────────────────────────────
BG          = "#FFFFFF"        # figure / axes background
PANEL_BG    = "#F7F8FA"        # subtle panel tint
BORDER      = "#D0D5DD"        # spine / grid lines
TEXT_PRI    = "#1A1D23"        # titles, annotations
TEXT_MUT    = "#6B7280"        # muted labels, watermark
ACCENT_BLUE = "#2563EB"
ACCENT_RED  = "#DC2626"
ACCENT_AMB  = "#D97706"
ACCENT_GRN  = "#16A34A"

STRESS_PAL  = {0: ACCENT_BLUE, 1: ACCENT_RED}

plt.rcParams.update({
    "figure.facecolor":    BG,
    "axes.facecolor":      PANEL_BG,
    "axes.edgecolor":      BORDER,
    "axes.labelcolor":     TEXT_PRI,
    "axes.titlecolor":     TEXT_PRI,
    "axes.grid":           True,
    "grid.color":          BORDER,
    "grid.linewidth":      0.55,
    "grid.alpha":          0.7,
    "grid.linestyle":      "--",
    "xtick.color":         TEXT_MUT,
    "ytick.color":         TEXT_MUT,
    "xtick.labelsize":     10,
    "ytick.labelsize":     10,
    "axes.titlesize":      12,
    "axes.titleweight":    "bold",
    "axes.titlepad":       10,
    "axes.labelsize":      10,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.spines.left":    True,
    "axes.spines.bottom":  True,
    "font.family":         "DejaVu Sans",
    "text.color":          TEXT_PRI,
    "legend.facecolor":    BG,
    "legend.edgecolor":    BORDER,
    "savefig.facecolor":   BG,
    "savefig.dpi":         180,
    "figure.dpi":          110,
})


# def _watermark(fig, text="Confidential Research"):
#     fig.text(0.99, 0.005, text, fontsize=7, color=TEXT_MUT,
#              ha="right", va="bottom", alpha=0.5, style="italic")


# ─────────────────────────────────────────────
#  1. LABEL DISTRIBUTIONS
# ─────────────────────────────────────────────
def plot_label_distributions(eda_df, labels=("stress_1d", "stress_3d", "stress_5d")):
    labels = [l for l in labels if l in eda_df.columns]
    horizon_map = {"stress_1d": "1-Day", "stress_3d": "3-Day", "stress_5d": "5-Day"}

    for col in labels:
        fig, ax = plt.subplots(figsize=(6.2, 5.4))
        fig.subplots_adjust(top=0.86, bottom=0.14, left=0.10, right=0.96)

        counts = eda_df[col].value_counts(dropna=True).sort_index()
        total = counts.sum()
        x_vals = np.arange(len(counts))
        colors = [
            STRESS_PAL.get(int(k), ACCENT_AMB) if pd.notna(k) else ACCENT_AMB
            for k in counts.index
        ]

        bars = ax.bar(
            x_vals,
            counts.values,
            color=colors,
            width=0.50,
            zorder=3,
            edgecolor="white",
            linewidth=0.8,
            alpha=0.88
        )

        for bar, val in zip(bars, counts.values):
            pct = val / total * 100
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + total * 0.012,
                f"{val:,}\n({pct:.1f}%)",
                ha="center",
                va="bottom",
                fontsize=10,
                color=TEXT_PRI,
                fontweight="bold"
            )

        ax.set_xticks(x_vals)
        ax.set_xticklabels(
            ["No Stress\n(0)" if k == 0 else "Stress\n(1)" for k in counts.index],
            fontsize=11
        )
        ax.set_ylabel("Count", labelpad=6)
        ax.set_title(f"{horizon_map.get(col, col)} Horizon")
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
        )
        ax.set_xlim(-0.6, len(counts) - 0.4)
        ax.set_ylim(0, counts.max() * 1.25)
        ax.spines["bottom"].set_color(BORDER)
        ax.spines["left"].set_color(BORDER)

        if 0 in counts.index and 1 in counts.index:
            ratio = counts[0] / counts[1]
            ax.text(
                0.97,
                0.97,
                f"Imbalance\n{ratio:.1f} : 1",
                transform=ax.transAxes,
                ha="right",
                va="top",
                fontsize=8.5,
                color=ACCENT_AMB,
                bbox=dict(
                    boxstyle="round,pad=0.4",
                    facecolor=BG,
                    edgecolor=ACCENT_AMB,
                    linewidth=0.9
                )
            )

        fig.suptitle(
            f"Stress Label Distribution - {horizon_map.get(col, col)} Forecast Horizon",
            fontsize=14,
            fontweight="bold",
            color=TEXT_PRI,
            y=0.97
        )

        outname = f"label_distribution_{col}.png"
        plt.savefig(f"../data/visuals/features/{outname}", bbox_inches="tight")
        plt.show()
        print(f"Saved → {outname}")


# ─────────────────────────────────────────────
#  2. CORRELATION HEATMAP
# ─────────────────────────────────────────────
def plot_correlation_heatmap(eda_df, candidate_features=None):
    if candidate_features is None:
        key_features = eda_df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        key_features = [c for c in candidate_features if c in eda_df.columns]

    corr = eda_df[key_features].corr()
    n = len(key_features)

    cmap = LinearSegmentedColormap.from_list(
        "academic_div",
        ["#2563EB", "#93C5FD", "#F3F4F6", "#FCA5A5", "#DC2626"],
        N=256
    )

    fig, ax = plt.subplots(figsize=(15, 12))
    fig.subplots_adjust(top=0.92, bottom=0.22, left=0.18, right=0.92)

    im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect="auto")

    for i in range(n):
        for j in range(n):
            val = corr.values[i, j]
            text_color = TEXT_PRI if abs(val) < 0.55 else "white"
            weight = "bold" if i != j and abs(val) > 0.5 else "normal"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=8, color=text_color, fontweight=weight)

    for k in np.arange(-0.5, n, 1):
        ax.axhline(k, color=BG, linewidth=1.4, zorder=5)
        ax.axvline(k, color=BG, linewidth=1.4, zorder=5)

    for d in range(n):
        ax.add_patch(plt.Rectangle(
            (d - 0.5, d - 0.5), 1, 1,
            fill=False, edgecolor=ACCENT_AMB,
            linewidth=0.9, zorder=6))

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(key_features, rotation=45, ha="right", fontsize=9, color=TEXT_PRI)
    ax.set_yticklabels(key_features, fontsize=9, color=TEXT_PRI)
    ax.tick_params(top=False, bottom=False, left=False, right=False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.ax.yaxis.set_tick_params(color=TEXT_MUT, labelsize=9)
    cbar.outline.set_edgecolor(BORDER)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_MUT)
    cbar.set_label("Pearson r", color=TEXT_MUT, fontsize=10, labelpad=8)

    title = (
        "Correlation Heatmap of All Base Features"
        if candidate_features is None
        else "Correlation Heatmap of Selected Base Features"
    )
    fig.suptitle(title, fontsize=16, fontweight="bold", color=TEXT_PRI, y=0.97)

    outname = (
        "correlation_heatmap_all_base.png"
        if candidate_features is None
        else "correlation_heatmap_selected_base.png"
    )
    plt.savefig(f"../data/visuals/features/{outname}", bbox_inches="tight")
    plt.show()
    print(f"Saved → {outname}")
    

# ─────────────────────────────────────────────
#  3. FEATURE DISTRIBUTIONS
# ─────────────────────────────────────────────
def plot_feature_distributions(eda_df, dist_features=None):
    from scipy.stats import gaussian_kde

    if dist_features is None:
        dist_features = ["es_range", "vix_close", "BBB_SPREAD", "sentiment_mean"]
    dist_features = [c for c in dist_features if c in eda_df.columns]

    FEAT_META = {
        "es_range":       {"label": "ES Range",          "unit": "pts", "color": ACCENT_BLUE},
        "vix_close":      {"label": "VIX Close",         "unit": "",    "color": ACCENT_RED},
        "BBB_SPREAD":     {"label": "BBB Credit Spread",  "unit": "%",   "color": ACCENT_AMB},
        "sentiment_mean": {"label": "Sentiment Mean",    "unit": "",    "color": ACCENT_GRN},
    }

    ncols = 2
    nrows = int(np.ceil(len(dist_features) / ncols))

    # plt.subplots + subplots_adjust gives precise control over row gap
    fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.8 * nrows))
    fig.subplots_adjust(top=0.88, bottom=0.09, left=0.08,
                        right=0.97, hspace=0.52, wspace=0.28)
    axes_flat = axes.flatten() if nrows > 1 else list(axes)

    for idx, feat in enumerate(dist_features):
        ax   = axes_flat[idx]
        meta = FEAT_META.get(feat, {"label": feat, "unit": "", "color": ACCENT_BLUE})
        data = eda_df[feat].dropna()
        col  = meta["color"]

        counts, edges = np.histogram(data, bins=45)
        bin_centers   = 0.5 * (edges[:-1] + edges[1:])
        norm_h        = counts / counts.max()

        # Gradient bars
        for x, cnt, norm, lo, hi in zip(
                bin_centers, counts, norm_h, edges[:-1], edges[1:]):
            rgba = plt.matplotlib.colors.to_rgba(col, alpha=0.25 + 0.60 * norm)
            ax.bar(x, cnt, width=(hi - lo) * 0.90,
                   color=rgba, zorder=3, linewidth=0)

        # KDE curve + fill
        kde     = gaussian_kde(data, bw_method="scott")
        x_line  = np.linspace(data.min(), data.max(), 400)
        k_scale = counts.max() / kde(x_line).max()
        ax.plot(x_line, kde(x_line) * k_scale,
                color=col, linewidth=2.2, zorder=5)
        ax.fill_between(x_line, kde(x_line) * k_scale,
                        alpha=0.10, color=col, zorder=4)

        # Stat reference lines
        ymax = counts.max()
        for val, lbl, ls, clr in [
            (data.mean(),         "Mean",   "--", TEXT_PRI),
            (data.median(),       "Median", ":",  ACCENT_AMB),
            (data.quantile(0.95), "P95",    "-.", ACCENT_RED),
        ]:
            ax.axvline(val, linestyle=ls, linewidth=1.2,
                       color=clr, alpha=0.80, zorder=6)
            ax.text(val, ymax * 0.97, f" {lbl}\n {val:.3g}",
                    fontsize=7.5, color=clr, va="top",
                    rotation=90, alpha=0.90)

        # Stats badge
        stats_txt = (
            f"n = {len(data):,}\n"
            f"μ = {data.mean():.3g}\n"
            f"σ = {data.std():.3g}\n"
            f"Skew = {data.skew():.2f}\n"
            f"Kurt = {data.kurtosis():.2f}"
        )
        ax.text(0.97, 0.97, stats_txt,
                transform=ax.transAxes,
                ha="right", va="top",
                fontsize=8, color=TEXT_MUT,
                fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.45",
                          facecolor=BG, edgecolor=BORDER,
                          linewidth=0.8, alpha=0.95))

        unit = f" ({meta['unit']})" if meta["unit"] else ""
        ax.set_title(meta["label"])
        ax.set_xlabel(f"{meta['label']}{unit}", labelpad=5)
        ax.set_ylabel("Frequency", labelpad=5)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
        ax.spines["bottom"].set_color(BORDER)
        ax.spines["left"].set_color(BORDER)
        ax.set_xlim(edges[0], edges[-1])
        ax.set_ylim(0, ymax * 1.20)

    # Hide any unused panels
    for ax in axes_flat[len(dist_features):]:
        ax.set_visible(False)

    fig.suptitle("Feature Distributions — Selected Base Signals",
                 fontsize=15, fontweight="bold", color=TEXT_PRI, y=0.96)
    
    plt.savefig("../data/visuals/features/feature_distributions.png", bbox_inches="tight")
    plt.show()
    print("Saved → feature_distributions.png")


# ─────────────────────────────────────────────
#  ENTRYPOINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    tmp = df[["date", "close"]].copy().sort_values("date").reset_index(drop=True)
    tmp["ret"]     = np.log(tmp["close"]).diff()
    tmp["vol_14d"] = tmp["ret"].rolling(14).std()
    q_ret = tmp["ret"].quantile(0.01)
    q_vol = tmp["vol_14d"].quantile(0.9)
    tmp["stress_t"] = ((tmp["ret"] <= q_ret) | (tmp["vol_14d"] >= q_vol)).astype(int)

    def horizon_any_future(stress, horizon):
        shifted = pd.concat([stress.shift(-i) for i in range(1, horizon + 1)], axis=1)
        return shifted.max(axis=1)

    tmp["stress_1d"] = horizon_any_future(tmp["stress_t"], 1)
    tmp["stress_3d"] = horizon_any_future(tmp["stress_t"], 3)

    eda_df = df_out.copy()
    eda_df = eda_df.merge(
        tmp[["date", "stress_1d", "stress_3d"]],
        on="date", how="left")

    plot_label_distributions(eda_df)

    USE_ALL_BASE_FEATURES = False

    if USE_ALL_BASE_FEATURES:
        plot_correlation_heatmap(eda_df)
    else:
        candidate_features = [
            "es_close", "es_range", "dxy_close", "US2Y", "US10Y", "FED_RATE_USD",
            "vix_close", "BBB_SPREAD", "TED_SPREAD", "sentiment_mean", "sentiment_std", "stress_ratio"
        ]
        plot_correlation_heatmap(eda_df, candidate_features)

    dist_features = ["es_range", "vix_close", "BBB_SPREAD", "sentiment_mean"]
    plot_feature_distributions(eda_df, dist_features)